# ZemaVoice Stage 13 — Female Amharic Voice Reference Preparation

Prepare an **authorized** female Amharic voice sample for a voice-cloning experiment. The uploaded sample contains speech and background music, so choose a speech-only segment. This notebook **does not clone voices by itself**; it produces a clean reference WAV for the Stage 11 OmniVoice test. Voice cloning is not guaranteed to work or sound natural. Do not upload recordings unless you have the speaker's permission.

Run cells in order. **CPU runtime is sufficient** for preparation.


In [ ]:
!pip -q install soundfile librosa ipywidgets


In [ ]:
from google.colab import files
from IPython.display import display, Audio
from pathlib import Path
import subprocess, json, os, soundfile as sf, numpy as np
uploaded=files.upload()
assert uploaded,'Please upload the female Amharic MP3 or WAV.'
source=next(iter(uploaded))
print('Uploaded:',source)
info=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',source],capture_output=True,text=True,check=True)
duration=float(info.stdout.strip())
print('Total duration:',round(duration,2),'seconds')


In [ ]:
import ipywidgets as widgets
start=widgets.FloatSlider(value=0,min=0,max=max(0,duration-1),step=0.25,description='Start (s):',continuous_update=False,layout=widgets.Layout(width='80%'))
end=widgets.FloatSlider(value=min(duration,17),min=1,max=duration,step=0.25,description='End (s):',continuous_update=False,layout=widgets.Layout(width='80%'))
display(widgets.VBox([start,end]))
print('Choose a section containing ONLY speech, without background music. For this recording, start by trying 0–17 seconds, then listen and adjust.')


In [ ]:
a,b=float(start.value),float(end.value)
assert 0<=a<b<=duration,'Choose valid start/end times.'
assert b-a>=5,'Use at least 5 seconds of clean speech.'
out='/content/zemavoice_stage13_female_reference.wav'
# Light filtering only; heavy denoising may distort speaker identity.
cmd=['ffmpeg','-y','-loglevel','error','-ss',str(a),'-to',str(b),'-i',source,'-ac','1','-ar','24000','-af','highpass=f=75,lowpass=f=9500',out]
p=subprocess.run(cmd,capture_output=True,text=True)
if p.returncode: raise RuntimeError(p.stderr[-2000:])
y,sr=sf.read(out)
print('Reference:',round(len(y)/sr,2),'seconds,',sr,'Hz')
display(Audio(y,rate=sr))
print('IMPORTANT: If you hear music, change Start/End and rerun this cell.')


In [ ]:
from google.colab import files
files.download('/content/zemavoice_stage13_female_reference.wav')


## Next: test the reference in Stage 11
Open the [Stage 11 OmniVoice notebook](https://colab.research.google.com/github/abrahamredi8/Zema-/blob/main/independent_engine/ZemaVoice_Stage11_OmniVoice_Amharic_Test.ipynb), enable **T4 GPU**, and upload `zemavoice_stage13_female_reference.wav` in its optional reference-audio cell. Stage 11 uses experimental third-party model code and **may require troubleshooting** if the published package interface has changed. Only proceed if you have the speaker's consent.

After testing, compare the output with the original speech for pronunciation, naturalness, and voice similarity. Do not treat it as a finished ZemaVoice voice until it passes listening tests.
